# 练习实验：线性回归

欢迎来到你的第一个练习实验！在这个实验中，你将实现单变量线性回归来预测餐饮特许经营的利润。


# 大纲
- [ 1 - 包 ](#1)
- [ 2 - 单变量线性回归 ](#2)
  - [ 2.1 问题陈述](#2.1)
  - [ 2.2  数据集](#2.2)
  - [ 2.3 线性回归复习](#2.3)
  - [ 2.4  计算代价](#2.4)
    - [ 练习 1](#ex01)
  - [ 2.5 梯度下降 ](#2.5)
    - [ 练习 2](#ex02)
  - [ 2.6 使用批量梯度下降学习参数 ](#2.6)


<a name="1"></a>
## 1 - 包 

首先，让我们运行下面的单元格以导入此作业所需的所有包。
- [numpy](www.numpy.org) 是在 Python 中使用矩阵的基本包。
- [matplotlib](http://matplotlib.org) 是 Python 中著名的绘图库。
- ``utils.py`` 包含此作业的辅助函数。你不需要修改此文件中的代码。


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from utils import *
import copy
import math
%matplotlib inline

## 2 -  问题陈述

假设你是餐饮特许经营的 CEO，正在考虑在不同的城市开设新的分店。
- 你想将业务扩展到可能给你的餐厅带来更高利润的城市。
- 该连锁店已经在各个城市拥有餐厅，并且你有来自这些城市的利润和人口数据。
- 你也有作为新餐厅候选城市的数据。
    - 对于这些城市，你有城市人口。
    
你能使用数据帮助你确定哪些城市可能给你的业务带来更高的利润吗？

## 3 - 数据集

你将从加载此任务的数据集开始。
- 下面显示的 `load_data()` 函数将数据加载到变量 `x_train` 和 `y_train` 中
  - `x_train` 是城市人口
  - `y_train` 是该城市餐厅的利润。负利润值表示亏损。
  - `X_train` 和 `y_train` 都是 numpy 数组。

In [ ]:
# load the dataset
x_train, y_train = load_data()

#### 查看变量
在开始任何任务之前，熟悉你的数据集通常很有用。
- 一个好的开始是打印出每个变量并查看其包含的内容。

下面的代码打印变量 `x_train` 和变量的类型。

In [ ]:
# print x_train
print("Type of x_train:",type(x_train))
print("First five elements of x_train are:\n", x_train[:5]) 

`x_train` 是一个包含所有大于零的小数值的 numpy 数组。
- 这些值代表城市人口乘以 10,000
- 例如，6.1101 意味着该城市的人口是 61,101
  
现在，让我们打印 `y_train`

In [ ]:
# print y_train
print("Type of y_train:",type(y_train))
print("First five elements of y_train are:\n", y_train[:5])  

类似地，`y_train` 是一个包含小数值的 numpy 数组，有些是负数，有些是正数。
- 这些代表你的餐厅在每个城市的平均月利润，单位为 \$10,000。
  - 例如，17.592 代表该城市平均月利润为 \$175,920。
  - -2.6807 代表该城市平均月亏损为 -\$26,807。

#### 检查变量的维度

熟悉数据的另一种有用方法是查看其维度。

请打印 `x_train` 和 `y_train` 的形状，看看你的数据集中有多少训练样本。

In [ ]:
print ('The shape of x_train is:', x_train.shape)
print ('The shape of y_train is: ', y_train.shape)
print ('Number of training examples (m):', len(x_train))

城市人口数组有 97 个数据点，月平均利润也有 97 个数据点。这些是 NumPy 1D 数组。

#### 可视化你的数据

通过可视化来理解数据通常很有用。
- 对于这个数据集，你可以使用散点图来可视化数据，因为它只有两个属性要绘制（利润和人口）。
- 你在现实生活中遇到的许多其他问题都有两个以上的属性（例如，人口、平均家庭收入、月利润、月销售额）。当你有两个以上的属性时，你仍然可以使用散点图来查看每对属性之间的关系。


In [ ]:
# Create a scatter plot of the data. To change the markers to red "x",
# we used the 'marker' and 'c' parameters
plt.scatter(x_train, y_train, marker='x', c='r') 

# Set the title
plt.title("Profits vs. Population per city")
# Set the y-axis label
plt.ylabel('Profit in $10,000')
# Set the x-axis label
plt.xlabel('Population of City in 10,000s')
plt.show()

你的目标是建立一个线性回归模型来拟合这些数据。
- 有了这个模型，你可以输入一个新城市的人口，并让模型估计该城市餐厅的潜在月利润。

<a name="4"></a>
## 4 - 线性回归复习

在这个练习实验中，你将把线性回归参数 $(w,b)$ 拟合到你的数据集。
- 线性回归的模型函数（即从 `x`（城市人口）映射到 `y`（该城市餐厅月利润）的函数）表示为 
    $$f_{w,b}(x) = wx + b$$
    

- 要训练线性回归模型，你想找到最适合你的数据集的 $(w,b)$ 参数。

    - 为了比较 $(w,b)$ 的一个选择比另一个选择好还是坏，你可以用代价函数 $J(w,b)$ 来评估它
      - $J$ 是 $(w,b)$ 的函数。也就是说，代价 $J(w,b)$ 的值取决于 $(w,b)$ 的值。
  
    - 最适合你的数据的 $(w,b)$ 的选择是具有最小代价 $J(w,b)$ 的那个。


- 要找到获得最小可能代价 $J(w,b)$ 的值 $(w,b)$，你可以使用一种称为 **梯度下降** 的方法。
  - 随着梯度下降的每一步，你的参数 $(w,b)$ 会更接近实现最低代价 $J(w,b)$ 的最优值。
  

- 训练好的线性回归模型然后可以接受输入特征 $x$（城市人口）并输出预测 $f_{w,b}(x)$（该城市餐厅的预测月利润）。

<a name="5"></a>
## 5 - 计算代价

梯度下降涉及重复步骤来调整参数 $(w,b)$ 的值，以逐渐获得越来越小的代价 $J(w,b)$。
- 在梯度下降的每一步，通过在 $(w,b)$ 更新时计算代价 $J(w,b)$ 来监控你的进度将很有帮助。
- 在本节中，你将实现一个函数来计算 $J(w,b)$，以便你可以检查梯度下降实现的进度。

#### 代价函数
正如你可能从讲座中回忆的那样，对于一个变量，线性回归的代价函数 $J(w,b)$ 定义为

$$J(w,b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)})^2$$ 

- 你可以将 $f_{w,b}(x^{(i)})$ 视为模型对你餐厅利润的预测，而 $y^{(i)}$ 是数据中记录的实际利润。
- $m$ 是数据集中的训练样本数量

#### 模型预测

- 对于单变量线性回归，模型 $f_{w,b}$ 对样本 $x^{(i)}$ 的预测表示为：

$$ f_{w,b}(x^{(i)}) = wx^{(i)} + b$$

这是直线的方程，截距为 $b$，斜率为 $w$

#### 实现

请完成下面的 `compute_cost()` 函数以计算代价 $J(w,b)$。

<a name="ex01"></a>
### 练习 1

完成下面的 `compute_cost` 以：

* 迭代训练样本，并为每个样本计算：
    * 该样本的模型的预测 
    $$
    f_{wb}(x^{(i)}) =  wx^{(i)} + b 
    $$
   
    * 该样本的代价  $$cost^{(i)} =  (f_{wb} - y^{(i)})^2$$
    

* 返回所有样本的总代价
$$J(\mathbf{w},b) = \frac{1}{2m} \sum\limits_{i = 0}^{m-1} cost^{(i)}$$
  * 这里，$m$ 是训练样本的数量，$\sum$ 是求和运算符

如果你卡住了，你可以查看下面单元格后给出的提示来帮助你实现。

In [ ]:
# UNQ_C1
# GRADED FUNCTION: compute_cost

def compute_cost(x, y, w, b): 
    """
    Computes the cost function for linear regression.
    
    Args:
        x (ndarray): Shape (m,) Input to the model (Population of cities) 
        y (ndarray): Shape (m,) Label (Actual profits for the cities)
        w, b (scalar): Parameters of the model
    
    Returns
        total_cost (float): The cost of using w,b as the parameters for linear regression
               to fit the data points in x and y
    """
    # number of training examples
    m = x.shape[0] 
    
    # You need to return this variable correctly
    total_cost = 0
    
    ### START CODE HERE ###  
    
    ### END CODE HERE ### 

    return total_cost

<details>
  <summary><font size="3" color="darkgreen"><b>点击获取提示</b></font></summary>
    
    
   * 你可以用代码表示求和运算符，例如：$h = \sum\limits_{i = 0}^{m-1} 2i$ 如下：
     ```python 
    h = 0
    for i in range(m):
        h = h + 2*i
    ```
  
   * 在这种情况下，你可以使用 for 循环迭代 `x` 中的所有样本，并将每次迭代的 `cost` 添加到循环外初始化的变量（`cost_sum`）中。

   * 然后，你可以将 `total_cost` 作为 `cost_sum` 除以 `2m` 返回。
     
    <details>
          <summary><font size="2" color="darkblue"><b> 点击获取更多提示</b></font></summary>
        
    * 这是你如何构建此函数的整体实现
    ```python 
    def compute_cost(x, y, w, b):
        # 训练样本的数量
        m = x.shape[0] 
    
        # 你需要正确返回此变量
        total_cost = 0
    
        ### START CODE HERE ###  
        # 跟踪每个样本的代价总和的变量
        cost_sum = 0
    
        # 循环遍历训练样本
        for i in range(m):
            # 你的代码在这里获取第 i 个样本的预测 f_wb
            f_wb = 
            # 你的代码在这里获取与第 i 个样本关联的代价
            cost = 
        
            # 添加到每个样本的代价总和
            cost_sum = cost_sum + cost 

        # 获取总代价为总和除以 (2*m)
        total_cost = (1 / (2 * m)) * cost_sum
        ### END CODE HERE ### 

        return total_cost
    ```
    
    如果你仍然卡住，你可以查看下面提供的提示，找出如何计算 `f_wb` 和 `cost`。
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 f_wb 的提示</b></font></summary>
           &emsp; &emsp; 对于标量 $a$, $b$ 和 $c$（<code>x[i]</code>, <code>w</code> 和 <code>b</code> 都是标量），你可以用代码计算方程 $h = ab + c$ 为 <code>h = a * b + c</code>
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 f 的更多提示</b></font></summary>
               &emsp; &emsp; 你可以计算 f_wb 为 <code>f_wb = w * x[i] + b </code>
           </details>
    </details>

     <details>
          <summary><font size="2" color="darkblue"><b>计算代价的提示</b></font></summary>
          &emsp; &emsp; 你可以计算变量 z 的平方为 z**2
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算代价的更多提示</b></font></summary>
              &emsp; &emsp; 你可以计算代价为 <code>cost = (f_wb - y[i]) ** 2</code>
          </details>
    </details>
        
    </details>

</details>

    


你可以通过运行以下测试代码来检查你的实现是否正确：

In [ ]:
# Compute cost with some initial values for paramaters w, b
initial_w = 2
initial_b = 1

cost = compute_cost(x_train, y_train, initial_w, initial_b)
print(type(cost))
print(f'Cost at initial w: {cost:.3f}')

# Public tests
from public_tests import *
compute_cost_test(compute_cost)

**预期输出**:
<table>
  <tr>
    <td> <b>初始 w 的代价:<b> 75.203 </td> 
  </tr>
</table>

<a name="6"></a>
## 6 - 梯度下降 

在本节中，你将为线性回归参数 $w, b$ 实现梯度。

正如讲座视频中所述，梯度下降算法是：

$$\begin{align*}& \text{重复直到收敛:} \; \lbrace \newline \; & \phantom {0000} b := b -  \alpha \frac{\partial J(w,b)}{\partial b} \newline       \; & \phantom {0000} w := w -  \alpha \frac{\partial J(w,b)}{\partial w} \tag{1}  \; & 
\newline & \rbrace\end{align*}$$

其中，参数 $w, b$ 是同时更新的，其中
$$
\frac{\partial J(w,b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) - y^{(i)}) \tag{2}
$$
$$
\frac{\partial J(w,b)}{\partial w}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} (f_{w,b}(x^{(i)}) -y^{(i)})x^{(i)} \tag{3}
$$
* m 是数据集中的训练样本数量

    
*  $f_{w,b}(x^{(i)})$ 是模型的预测，而 $y^{(i)}$ 是目标值


你将实现一个名为 `compute_gradient` 的函数，该函数计算 $\frac{\partial J(w)}{\partial w}$, $\frac{\partial J(w)}{\partial b}$

<a name="ex02"></a>
### 练习 2

请完成 `compute_gradient` 函数以：

* 迭代训练样本，并为每个样本计算：
    * 该样本的模型的预测 
    $$
    f_{wb}(x^{(i)}) =  wx^{(i)} + b 
    $$
   
    * 来自该样本的参数 $w, b$ 的梯度 
        $$
        \frac{\partial J(w,b)}{\partial b}^{(i)}  =  (f_{w,b}(x^{(i)}) - y^{(i)}) 
        $$
        $$
        \frac{\partial J(w,b)}{\partial w}^{(i)}  =  (f_{w,b}(x^{(i)}) -y^{(i)})x^{(i)} 
        $$
    

* 返回所有样本的总梯度更新
    $$
    \frac{\partial J(w,b)}{\partial b}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} \frac{\partial J(w,b)}{\partial b}^{(i)}
    $$
    
    $$
    \frac{\partial J(w,b)}{\partial w}  = \frac{1}{m} \sum\limits_{i = 0}^{m-1} \frac{\partial J(w,b)}{\partial w}^{(i)} 
    $$
  * 这里，$m$ 是训练样本的数量，$\sum$ 是求和运算符

如果你卡住了，你可以查看下面单元格后给出的提示来帮助你实现。

In [ ]:
# UNQ_C2
# GRADED FUNCTION: compute_gradient
def compute_gradient(x, y, w, b): 
    """
    Computes the gradient for linear regression 
    Args:
      x (ndarray): Shape (m,) Input to the model (Population of cities) 
      y (ndarray): Shape (m,) Label (Actual profits for the cities)
      w, b (scalar): Parameters of the model  
    Returns
      dj_dw (scalar): The gradient of the cost w.r.t. the parameters w
      dj_db (scalar): The gradient of the cost w.r.t. the parameter b     
     """
    
    # Number of training examples
    m = x.shape[0]
    
    # You need to return the following variables correctly
    dj_dw = 0
    dj_db = 0
    
    ### START CODE HERE ### 
    
    ### END CODE HERE ### 
        
    return dj_dw, dj_db

<details>
  <summary><font size="3" color="darkgreen"><b>点击获取提示</b></font></summary>
       
    * 你可以用代码表示求和运算符，例如：$h = \sum\limits_{i = 0}^{m-1} 2i$ 如下：
     ```python 
    h = 0
    for i in range(m):
        h = h + 2*i
    ```
    
    * 在这种情况下，你可以使用 for 循环迭代 `x` 中的所有样本，并为每个样本，将该样本的梯度添加到在循环外初始化的变量 `dj_dw` 和 `dj_db` 中。

   * 然后，你可以返回 `dj_dw` 和 `dj_db`，都除以 `m`。
    <details>
          <summary><font size="2" color="darkblue"><b> 点击获取更多提示</b></font></summary>
        
    * 这是你如何构建此函数的整体实现
    ```python 
    def compute_gradient(x, y, w, b): 
        """
        Computes the gradient for linear regression 
        Args:
          x (ndarray): Shape (m,) Input to the model (Population of cities) 
          y (ndarray): Shape (m,) Label (Actual profits for the cities)
          w, b (scalar): Parameters of the model  
        Returns
          dj_dw (scalar): The gradient of the cost w.r.t. the parameters w
          dj_db (scalar): The gradient of the cost w.r.t. the parameter b     
         """
    
        # 训练样本的数量
        m = x.shape[0]
    
        # 你需要正确返回以下变量
        dj_dw = 0
        dj_db = 0
    
        ### START CODE HERE ### 
        # 循环遍历样本
        for i in range(m):  
            # 你的代码在这里获取第 i 个样本的预测 f_wb
            f_wb = 
            
            # 你的代码在这里获取来自第 i 个样本的 w 的梯度 
            dj_dw_i = 
        
            # 你的代码在这里获取来自第 i 个样本的 b 的梯度 
            dj_db_i = 
     
            # 更新 dj_db ：在 Python 中，a += 1 与 a = a + 1 相同
            dj_db += dj_db_i
        
            # 更新 dj_dw
            dj_dw += dj_dw_i
    
        # 将 dj_dw 和 dj_db 都除以 m
        dj_dw = dj_dw / m
        dj_db = dj_db / m
        ### END CODE HERE ### 
        
        return dj_dw, dj_db
    ```
    
    如果你仍然卡住，你可以查看下面提供的提示，找出如何计算 `f_wb` 和 `cost`。
    
    <details>
          <summary><font size="2" color="darkblue"><b>计算 f_wb 的提示</b></font></summary>
           &emsp; &emsp; 你在上一个练习中做过这个！对于标量 $a$, $b$ 和 $c$（<code>x[i]</code>, <code>w</code> 和 <code>b</code> 都是标量），你可以用代码计算方程 $h = ab + c$ 为 <code>h = a * b + c</code>
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 f 的更多提示</b></font></summary>
               &emsp; &emsp; 你可以计算 f_wb 为 <code>f_wb = w * x[i] + b </code>
           </details>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>计算 dj_dw_i 的提示</b></font></summary>
           &emsp; &emsp; 对于标量 $a$, $b$ 和 $c$（<code>f_wb</code>, <code>y[i]</code> 和 <code>x[i]</code> 都是标量），你可以用代码计算方程 $h = (a - b)c$ 为 <code>h = (a-b)*c</code>
          <details>
              <summary><font size="2" color="blue"><b>&emsp; &emsp; 计算 f 的更多提示</b></font></summary>
               &emsp; &emsp; 你可以计算 dj_dw_i 为 <code>dj_dw_i = (f_wb - y[i]) * x[i] </code>
           </details>
    </details>
        
    <details>
          <summary><font size="2" color="darkblue"><b>计算 dj_db_i 的提示</b></font></summary>
             &emsp; &emsp; 你可以计算 dj_db_i 为 <code> dj_db_i = f_wb - y[i] </code>
    </details>
        
    </details>

</details>

    


运行下面的单元格以使用参数 $w$,$b$ 的两种不同初始化来检查 `compute_gradient` 函数的实现。

In [ ]:
# Compute and display gradient with w initialized to zeroes
initial_w = 0
initial_b = 0

tmp_dj_dw, tmp_dj_db = compute_gradient(x_train, y_train, initial_w, initial_b)
print('Gradient at initial w, b (zeros):', tmp_dj_dw, tmp_dj_db)

compute_gradient_test(compute_gradient)

现在让我们在我们的数据集上运行上面实现的梯度下降算法。

**预期输出**:
<table>
  <tr>
    <td> <b>初始 , b (零) 时的梯度<b></td>
    <td> -65.32884975 -5.83913505154639</td> 
  </tr>
</table>

In [ ]:
# Compute and display cost and gradient with non-zero w
test_w = 0.2
test_b = 0.2
tmp_dj_dw, tmp_dj_db = compute_gradient(x_train, y_train, test_w, test_b)

print('Gradient at test w, b:', tmp_dj_dw, tmp_dj_db)

**预期输出**:
<table>
  <tr>
    <td> <b>测试 w 时的梯度<b></td>
    <td> -47.41610118 -4.007175051546391</td> 
  </tr>
</table>

<a name="2.6"></a>
### 2.6 使用批量梯度下降学习参数 

你现在将使用批量梯度下降找到线性回归模型的最优参数。回想一下，批量是指在一次迭代中运行所有样本。
- 你不需要为此部分实现任何内容。只需运行下面的单元格。

- 验证梯度下降是否正常工作的一个好方法是查看 $J(w,b)$ 的值，并检查它是否随着每一步而减小。

- 假设你正确实现了梯度并计算了代价，并且你有适当的学习率 alpha 值，$J(w,b)$ 应该永远不会增加，并且应该在算法结束时收敛到一个稳定值。

In [ ]:
def gradient_descent(x, y, w_in, b_in, cost_function, gradient_function, alpha, num_iters): 
    """
    Performs batch gradient descent to learn theta. Updates theta by taking 
    num_iters gradient steps with learning rate alpha
    
    Args:
      x :    (ndarray): Shape (m,)
      y :    (ndarray): Shape (m,)
      w_in, b_in : (scalar) Initial values of parameters of the model
      cost_function: function to compute cost
      gradient_function: function to compute the gradient
      alpha : (float) Learning rate
      num_iters : (int) number of iterations to run gradient descent
    Returns
      w : (ndarray): Shape (1,) Updated values of parameters of the model after
          running gradient descent
      b : (scalar)                Updated value of parameter of the model after
          running gradient descent
    """
    
    # number of training examples
    m = len(x)
    
    # An array to store cost J and w's at each iteration — primarily for graphing later
    J_history = []
    w_history = []
    w = copy.deepcopy(w_in)  #avoid modifying global w within function
    b = b_in
    
    for i in range(num_iters):

        # Calculate the gradient and update the parameters
        dj_dw, dj_db = gradient_function(x, y, w, b )  

        # Update Parameters using w, b, alpha and gradient
        w = w - alpha * dj_dw               
        b = b - alpha * dj_db               

        # Save cost J at each iteration
        if i<100000:      # prevent resource exhaustion 
            cost =  cost_function(x, y, w, b)
            J_history.append(cost)

        # Print cost every at intervals 10 times or as many iterations if < 10
        if i% math.ceil(num_iters/10) == 0:
            w_history.append(w)
            print(f"Iteration {i:4}: Cost {float(J_history[-1]):8.2f}   ")
        
    return w, b, J_history, w_history #return w and J,w history for graphing

现在让我们运行上面的梯度下降算法来学习我们数据集的参数。

In [ ]:
# initialize fitting parameters. Recall that the shape of w is (n,)
initial_w = 0.
initial_b = 0.

# some gradient descent settings
iterations = 1500
alpha = 0.01

w,b,_,_ = gradient_descent(x_train ,y_train, initial_w, initial_b, 
                     compute_cost, compute_gradient, alpha, iterations)
print("w,b found by gradient descent:", w, b)

**预期输出**:
<table>
  <tr>
    <td> <b> 梯度下降找到的 w, b<b></td>
    <td> 1.16636235 -3.63029143940436</td> 
  </tr>
</table>

我们现在将使用梯度下降的最终参数来绘制线性拟合。

回想一下，我们可以得到单个样本的预测 $f(x^{(i)})= wx^{(i)}+b$。

要计算整个数据集的预测，我们可以循环遍历所有训练样本并计算每个样本的预测。这显示在下面的代码块中。

In [ ]:
m = x_train.shape[0]
predicted = np.zeros(m)

for i in range(m):
    predicted[i] = w * x_train[i] + b

我们现在将绘制预测值以查看线性拟合。

In [ ]:
# Plot the linear fit
plt.plot(x_train, predicted, c = "b")

# Create a scatter plot of the data. 
plt.scatter(x_train, y_train, marker='x', c='r') 

# Set the title
plt.title("Profits vs. Population per city")
# Set the y-axis label
plt.ylabel('Profit in $10,000')
# Set the x-axis label
plt.xlabel('Population of City in 10,000s')

你的 $w,b$ 的最终值也可以用来预测利润。让我们预测 35,000 和 70,000 人口地区的利润。

- 模型接受以 10,000 为单位的城市人口作为输入。

- 因此，35,000 人可以转换为模型的输入 `np.array([3.5])`

- 同样，70,000 人可以转换为模型的输入 `np.array([7.])`


In [ ]:
predict1 = 3.5 * w + b
print('For population = 35,000, we predict a profit of $%.2f' % (predict1*10000))

predict2 = 7.0 * w + b
print('For population = 70,000, we predict a profit of $%.2f' % (predict2*10000))

**预期输出**:
<table>
  <tr>
    <td> <b> 对于人口 = 35,000，我们预测利润为<b></td>
    <td> $4519.77 </td> 
  </tr>
  
  <tr>
    <td> <b> 对于人口 = 70,000，我们预测利润为<b></td>
    <td> $45342.45 </td> 
  </tr>
</table>